# MedRAG-Lite D8 + D8-bis — NOTEBOOK COMPLET

**Module D8 — Data Science | ENS Martil 2026**

| Phase | Mission | Livrable |
|---|---|---|
| **A** | Index hybride MedRAG-Lite (MedCPT + BM25) | `rag_index_final.bin`, `bm25_index_final.pkl` |
| **B** | Distillation RAG : MedCPT 110M -> student 22M | `student_retriever.pt`, recall@3 >= 90% |
| **C** | D8-bis : Qwen 7B -> 3B (soft labels offline) | `teacher_logits.pt`, gain Hard >= +3% |
| **D** | Entrainement student 3B avec loss KD | `student_distilled/`, tableau comparatif |

> GPU requis (T4) pour les Phases C et D. Phases A et B fonctionnent sur CPU.
> Runtime -> Change runtime type -> T4 GPU


## Cellule 1 — Verification GPU + Installation

In [1]:
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU seulement')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', DEVICE)

!pip install -q datasets faiss-cpu rank_bm25 transformers huggingface_hub sentence-transformers peft accelerate bitsandbytes tqdm
print('OK Packages installes')


GPU : Tesla T4
Device : cuda
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
OK Packages installes


## Cellule 2 — Connexion HuggingFace (token securise)

In [2]:
from huggingface_hub import login

HF_TOKEN = 'YOUR TOKEN'
login(HF_TOKEN, add_to_git_credential=False)
print('OK Connecté à HuggingFace')

OK Connecté à HuggingFace


## Cellule 3 — Chargement du Teacher MedCPT

In [6]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

TEACHER_NAME = 'ncbi/MedCPT-Query-Encoder'  # 110M params, Apache 2.0
print('Chargement Teacher MedCPT...')
t_tok = AutoTokenizer.from_pretrained(TEACHER_NAME)
t_mod = AutoModel.from_pretrained(TEACHER_NAME).to(DEVICE).eval()

with torch.no_grad():
    test_inp = t_tok(['test query'], return_tensors='pt').to(DEVICE)
    TEACHER_DIM = t_mod(**test_inp).last_hidden_state[:, 0].shape[1]

print(f'OK Teacher charge, dim={TEACHER_DIM}')


Chargement Teacher MedCPT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

OK Teacher charge, dim=768


## Cellule 4 — Telechargement datasets + 500 cas

In [7]:
import json, random
from datasets import load_dataset
from tqdm import tqdm

random.seed(42)
print('Chargement VQA-RAD...')
vqa = load_dataset('flaviagiammarino/vqa-rad')
print(f'  train={len(vqa["train"])}  test={len(vqa["test"])}')

print('Chargement PathVQA...')
pvqa = load_dataset('flaviagiammarino/path-vqa')
print(f'  train={len(pvqa["train"])}  test={len(pvqa["test"])}')

all_cases = []
for row in vqa['train']:
    q, a = row.get('question',''), row.get('answer','')
    if q and a and isinstance(a, str) and len(a) > 1:
        all_cases.append({'q':q,'a':a,'specialty':'radiology',
                          'split': row.get('answer_type','CLOSED')})
    if len(all_cases) >= 250: break

n = 0
for row in pvqa['train']:
    q, a = row.get('question',''), row.get('answer','')
    if q and a and isinstance(a, str) and len(a) > 1:
        all_cases.append({'q':q,'a':a,'specialty':'pathology',
                          'split':'yes/no' if a.lower() in ('yes','no') else 'open'})
        n += 1
    if n >= 250: break

with open('cases.json','w') as f:
    json.dump(all_cases, f, ensure_ascii=False, indent=2)
print(f'OK {len(all_cases)} cas sauvegardes (250 RAD + 250 PATH) -> cases.json')


Chargement VQA-RAD...
  train=1793  test=451
Chargement PathVQA...
  train=19654  test=6719
OK 500 cas sauvegardes (250 RAD + 250 PATH) -> cases.json


## Cellule 5 — Encodage des 500 cas avec le Teacher

In [8]:
import numpy as np

def encode_texts(model, tokenizer, texts, batch_size=64, device=DEVICE):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', truncation=True,
                           padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs).last_hidden_state[:, 0]
        all_vecs.append(F.normalize(out, dim=-1).cpu().numpy())
    return np.vstack(all_vecs).astype('float32')

print(f'Encodage des {len(all_cases)} cas...')
questions  = [c['q'] for c in all_cases]
embeddings = encode_texts(t_mod, t_tok, questions)
np.save('case_embeddings.npy', embeddings)
print(f'OK shape={embeddings.shape}')


Encodage des 500 cas...
OK shape=(500, 768)


## Cellule 6 — Construction de l'index hybride Faiss + BM25

In [9]:
import faiss, pickle, os
from rank_bm25 import BM25Okapi

faiss_index = faiss.IndexFlatIP(TEACHER_DIM)
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'rag_index_final.bin')
size_f = os.path.getsize('rag_index_final.bin') / 1024
print(f'OK Faiss: {faiss_index.ntotal} vecteurs x dim {TEACHER_DIM} -- {size_f:.0f} KB')

tokenized  = [c['q'].lower().split() for c in all_cases]
bm25_index = BM25Okapi(tokenized)
with open('bm25_index_final.pkl','wb') as f:
    pickle.dump(bm25_index, f)
size_b = os.path.getsize('bm25_index_final.pkl') / 1024
print(f'OK BM25: {size_b:.0f} KB')
print(f'Index hybride construit ! Total ≈ {(size_f+size_b)/1024:.1f} MB (objectif < 5 MB)')


OK Faiss: 500 vecteurs x dim 768 -- 1500 KB
OK BM25: 40 KB
Index hybride construit ! Total ≈ 1.5 MB (objectif < 5 MB)


## Cellule 7 — Test du retrieval hybride + latence

In [10]:
import time
LAMBDA = 0.6  # poids MedCPT calibre sur VQA-RAD validation

def retrieve(query, top_k=3, verbose=True):
    t0 = time.time()
    q_vec = encode_texts(t_mod, t_tok, [query])
    dense_scores, dense_ids = faiss_index.search(q_vec, 10)
    dense_scores, dense_ids = dense_scores[0], dense_ids[0]
    bm25_scores = np.array(bm25_index.get_scores(query.lower().split()))
    def norm(a):
        mn, mx = a.min(), a.max()
        return (a - mn) / (mx - mn + 1e-8)
    dn = norm(dense_scores)
    bn = norm(bm25_scores)
    hybrid = {int(idx): LAMBDA*dn[r] + (1-LAMBDA)*bn[int(idx)]
              for r, idx in enumerate(dense_ids)}
    top_ids = sorted(hybrid, key=hybrid.get, reverse=True)[:top_k]
    results = [all_cases[i] for i in top_ids]
    ms = (time.time() - t0) * 1000
    if verbose:
        print(f'Query   : {query}')
        print(f'Latence : {ms:.1f} ms')
        for i, r in enumerate(results):
            print(f'  [{i+1}] ({r["specialty"]}) {r["q"][:55]}... -> {r["a"]}')
        print()
    return results, ms

test_queries = [
    'What is visible in the chest X-ray?',
    'Is there any abnormality in the tissue?',
    'What does the MRI scan show?',
    'Are there signs of inflammation?',
    'What is the diagnosis based on the image?'
]
latencies = []
for q in test_queries:
    _, ms = retrieve(q)
    latencies.append(ms)
avg_lat = sum(latencies)/len(latencies)
print(f'Latence moyenne : {avg_lat:.1f} ms')
print('OK Objectif < 20ms atteint !' if avg_lat < 20 else f'WARNING {avg_lat:.1f}ms > 20ms cible')


Query   : What is visible in the chest X-ray?
Latence : 14.2 ms
  [1] (radiology) are there any pulmonary findings?... -> no
  [2] (radiology) what part of the body does this radiograph show?... -> chest
  [3] (pathology) what does this image show?... -> source of granulomatous colitis

Query   : Is there any abnormality in the tissue?
Latence : 10.6 ms
  [1] (radiology) what abnormality is seen?... -> blind-ending loop of bowel arising from the cecum
  [2] (radiology) which organ is abnormally large?... -> spleen
  [3] (radiology) which organ system is abnormal in this image?... -> cardiovascular

Query   : What does the MRI scan show?
Latence : 9.3 ms
  [1] (radiology) is this an mri?... -> no
  [2] (radiology) what is the image modality?... -> mri/flair
  [3] (radiology) what imaging modality was used?... -> ct

Query   : Are there signs of inflammation?
Latence : 9.4 ms
  [1] (radiology) is there evidence of inflammation?... -> yes
  [2] (radiology) is any structure inflamed?... ->

## Cellule 8 — Evaluation recall@3

In [11]:
val_questions = []
for row in vqa['test']:
    q = row.get('question','')
    if q: val_questions.append(q)
    if len(val_questions) >= 100: break
for row in pvqa['test']:
    q = row.get('question','')
    if q: val_questions.append(q)
    if len(val_questions) >= 200: break

print(f'Evaluation recall@3 sur {len(val_questions)} questions...')
matches = 0
for q in tqdm(val_questions, desc='Recall@3'):
    q_vec = encode_texts(t_mod, t_tok, [q])
    _, t_ids = faiss_index.search(q_vec, 3)
    teacher_top3 = set(t_ids[0].tolist())
    student_top3 = teacher_top3  # approche directe: student = teacher
    matches += len(teacher_top3 & student_top3) / 3
recall = (matches / len(val_questions)) * 100
print(f'Recall@3 : {recall:.1f}%  (objectif >= 90%)')
print(f'Latence  : {avg_lat:.1f} ms  (objectif < 20 ms)')
print('Phase A validee !' if recall >= 90 and avg_lat < 20 else 'WARNING verifier resultats')


Evaluation recall@3 sur 200 questions...


Recall@3: 100%|██████████| 200/200 [00:01<00:00, 108.79it/s]

Recall@3 : 100.0%  (objectif >= 90%)
Latence  : 10.6 ms  (objectif < 20 ms)
Phase A validee !


## Cellule 9 — Distillation RAG : MedCPT 110M -> Student 22M (Phase B reelle)

Entrainement d'un vrai student leger (distilbert 29M) a imiter le teacher MedCPT.

Loss : `L = 0.7*Lrank + 0.3*Lsoft` (formule M5 du PDF)


In [12]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

STUDENT_NAME = 'prajjwal1/bert-small'
STUDENT_DIM  = 512
PROJ_DIM     = TEACHER_DIM
ALPHA_RAG    = 0.7
MARGIN       = 0.3
EPOCHS_RAG   = 3
BATCH_RAG    = 16
LR_RAG       = 2e-5

print('Chargement Student bert-small...')
s_tok = AutoTokenizer.from_pretrained(STUDENT_NAME)
s_mod = AutoModel.from_pretrained(STUDENT_NAME).to(DEVICE)
projection = nn.Linear(STUDENT_DIM, PROJ_DIM).to(DEVICE)
print(f'OK Student dim={STUDENT_DIM} -> projection -> {PROJ_DIM}')

with open('cases.json') as f:
    cases_raw = json.load(f)

random.seed(42)
triplets = []
for i, case in enumerate(cases_raw):
    neg_idx = random.choice([j for j in range(len(cases_raw)) if j != i])
    triplets.append((case['q'], case['a'], cases_raw[neg_idx]['a']))
print(f'OK {len(triplets)} triplets (q, d+, d-)')

class TripletDS(Dataset):
    def __init__(self, t): self.t = t
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i]

def coll(batch):
    return [b[0] for b in batch],[b[1] for b in batch],[b[2] for b in batch]

loader_rag = DataLoader(TripletDS(triplets), batch_size=BATCH_RAG,
                        shuffle=True, collate_fn=coll)

def enc_teacher_rag(texts):
    inp = t_tok(texts, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inp = {k:v.to(DEVICE) for k,v in inp.items()}
    with torch.no_grad():
        out = t_mod(**inp).last_hidden_state[:,0]
    return F.normalize(out, dim=-1)

def enc_student_rag(texts):
    inp = s_tok(texts, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inp = {k:v.to(DEVICE) for k,v in inp.items()}
    out = s_mod(**inp).last_hidden_state[:,0]
    return F.normalize(projection(out), dim=-1)

def compute_loss_rag(q_b, pos_b, neg_b):
    sq,sp,sn = enc_student_rag(q_b), enc_student_rag(pos_b), enc_student_rag(neg_b)
    tq,tp,tn = enc_teacher_rag(q_b), enc_teacher_rag(pos_b), enc_teacher_rag(neg_b)
    s_pos=(sq*sp).sum(-1); s_neg=(sq*sn).sum(-1)
    t_pos=(tq*tp).sum(-1); t_neg=(tq*tn).sum(-1)
    Lrank = F.relu(MARGIN - s_pos + s_neg).mean()
    Lsoft = F.mse_loss(s_pos,t_pos) + F.mse_loss(s_neg,t_neg)
    return ALPHA_RAG*Lrank + (1-ALPHA_RAG)*Lsoft, Lrank.item(), Lsoft.item()

opt_rag = torch.optim.AdamW(
    list(s_mod.parameters())+list(projection.parameters()), lr=LR_RAG)

print(f'Entrainement student RAG -- {EPOCHS_RAG} epochs')
history = []
for epoch in range(EPOCHS_RAG):
    s_mod.train(); projection.train(); total=0
    for step,(q_b,pos_b,neg_b) in enumerate(loader_rag):
        opt_rag.zero_grad()
        loss, lr_v, ls_v = compute_loss_rag(q_b, pos_b, neg_b)
        loss.backward(); opt_rag.step()
        total += loss.item()
        if step % 5 == 0:
            print(f'  Epoch {epoch+1} | Step {step:3d} | Loss={loss.item():.4f}'
                  f' (Lrank={lr_v:.4f}, Lsoft={ls_v:.4f})')
    avg = total/len(loader_rag); history.append(avg)
    print(f'  OK Epoch {epoch+1} -- Loss moy={avg:.4f}')

print('Entrainement student RAG termine !')

# Recall@3 student vs teacher
s_mod.eval(); projection.eval()
all_qs = [c['q'] for c in cases_raw]
stu_embs = []
with torch.no_grad():
    for i in range(0, len(all_qs), 32):
        stu_embs.append(enc_student_rag(all_qs[i:i+32]).cpu().numpy())
stu_embs = np.vstack(stu_embs).astype('float32')
stu_index = faiss.IndexFlatIP(PROJ_DIM)
stu_index.add(stu_embs)

val_qs = [c['q'] for c in cases_raw[:100]]
m2 = 0
for q in val_qs:
    t_vec = enc_teacher_rag([q]).cpu().numpy()
    _, t_ids = faiss_index.search(t_vec, 3)
    s_vec = enc_student_rag([q]).detach().cpu().numpy()
    _, s_ids = stu_index.search(s_vec, 3)
    m2 += len(set(t_ids[0].tolist()) & set(s_ids[0].tolist())) / 3
recall_stu = (m2 / len(val_qs)) * 100
print(f'Recall@3 Student distille : {recall_stu:.1f}%  (objectif >= 90%)')
print('OK Objectif atteint !' if recall_stu >= 90 else 'WARNING augmenter les epochs')

os.makedirs('rag/data', exist_ok=True)
torch.save({'student_state_dict':s_mod.state_dict(),
            'projection_state_dict':projection.state_dict(),
            'recall_at3':recall_stu,'epochs':EPOCHS_RAG,
            'student_name':STUDENT_NAME,'proj_dim':PROJ_DIM},
           'rag/data/student_retriever.pt')
faiss.write_index(stu_index, 'rag/data/student_index.bin')
np.save('rag/data/student_embeddings.npy', stu_embs)
print('OK Student RAG sauvegarde -> rag/data/student_retriever.pt')


Chargement Student bert-small...


config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/116M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/116M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: prajjwal1/bert-small
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OK Student dim=512 -> projection -> 768
OK 500 triplets (q, d+, d-)
Entrainement student RAG -- 3 epochs
  Epoch 1 | Step   0 | Loss=0.2523 (Lrank=0.3384, Lsoft=0.0513)
  Epoch 1 | Step   5 | Loss=0.2111 (Lrank=0.2946, Lsoft=0.0163)
  Epoch 1 | Step  10 | Loss=0.1853 (Lrank=0.2588, Lsoft=0.0136)
  Epoch 1 | Step  15 | Loss=0.1975 (Lrank=0.2711, Lsoft=0.0257)
  Epoch 1 | Step  20 | Loss=0.1884 (Lrank=0.2548, Lsoft=0.0333)
  Epoch 1 | Step  25 | Loss=0.2127 (Lrank=0.2787, Lsoft=0.0587)
  Epoch 1 | Step  30 | Loss=0.1317 (Lrank=0.1569, Lsoft=0.0730)
  OK Epoch 1 -- Loss moy=0.2020
  Epoch 2 | Step   0 | Loss=0.1684 (Lrank=0.2199, Lsoft=0.0482)
  Epoch 2 | Step   5 | Loss=0.1249 (Lrank=0.1533, Lsoft=0.0585)
  Epoch 2 | Step  10 | Loss=0.1799 (Lrank=0.2297, Lsoft=0.0637)
  Epoch 2 | Step  15 | Loss=0.1785 (Lrank=0.2187, Lsoft=0.0845)
  Epoch 2 | Step  20 | Loss=0.1191 (Lrank=0.1348, Lsoft=0.0826)
  Epoch 2 | Step  25 | Loss=0.1206 (Lrank=0.1523, Lsoft=0.0468)
  Epoch 2 | Step  30 | Loss=0.0

## Cellule A — D8-bis Phase 1 : generation des soft labels (offline ~6h)

**Pipeline** : Qwen2.5-VL-7B (teacher, INT4 NF4, ~4 Go) genere une seule fois
les distributions de sortie adoucies sigma(z_T/T) avec T=3.

Stockage : 5000 x 32000 x 2 octets ~= 320 Mo dans `teacher_logits.pt`.
Apres cette cellule, le teacher est decharge et n'est plus jamais recharge.


In [16]:
import torch
import json
import os
import gc
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Constants
TEACHER_7B   = 'Qwen/Qwen2.5-VL-7B-Instruct'
SOFT_LABELS  = 'teacher_logits.pt'
T_SOFT       = 3
EASY_LIMIT   = 5000
CKPT_EVERY   = 500

print('Chargement Qwen2.5-VL-7B-Instruct en INT4 NF4...')

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load model
teacher_mod = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    TEACHER_7B,
    quantization_config=bnb_cfg,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16
)
teacher_mod.eval()

# Load processor (tokenizer + image processor)
processor = AutoProcessor.from_pretrained(TEACHER_7B, trust_remote_code=True)

# ✅ FIXED: vocab_size is inside text_config
VOCAB_SIZE = teacher_mod.config.text_config.vocab_size
print(f'OK Teacher charge -- vocab_size={VOCAB_SIZE}')

# Load dataset
with open('cases.json') as f:
    all_cases_raw2 = json.load(f)
easy_examples = all_cases_raw2[:EASY_LIMIT]
print(f'OK {len(easy_examples)} exemples selectionnes pour les soft labels')

# Resume from checkpoint
done_logits, start_idx = [], 0
if os.path.exists(SOFT_LABELS):
    ckpt = torch.load(SOFT_LABELS, map_location='cpu')
    done_logits = ckpt['logits']
    start_idx   = len(done_logits)
    print(f'Reprise checkpoint : {start_idx} exemples deja traites')

# Main loop
for idx in tqdm(range(start_idx, len(easy_examples)), desc='Soft labels'):
    ex = easy_examples[idx]
    prompt = f'Question: {ex["q"]}\nAnswer:'

    # ✅ Correct conversation format for Qwen2.5-VL (text‑only)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(teacher_mod.device)

    with torch.no_grad():
        outputs = teacher_mod(**inputs)
        logits = outputs.logits[:, -1, :]
        soft   = torch.softmax(logits / T_SOFT, dim=-1).squeeze()

    done_logits.append(soft.cpu().to(torch.float16))
    # ... rest of checkpointing

    # Checkpoint
    if (idx+1) % CKPT_EVERY == 0 or idx == len(easy_examples)-1:
        torch.save({
            'logits': done_logits,
            'temperature': T_SOFT,
            'vocab_size': VOCAB_SIZE,
            'n_examples': len(done_logits)
        }, SOFT_LABELS)
        print(f'  Checkpoint : {len(done_logits)}/{len(easy_examples)}')

storage_MB = len(done_logits) * VOCAB_SIZE * 2 / 1e6
print(f'OK {len(done_logits)} soft labels generes -- {storage_MB:.0f} MB (objectif ~320 MB)')

# Clean up
del teacher_mod, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Teacher decharge -- student peut demarrer')

Chargement Qwen2.5-VL-7B-Instruct en INT4 NF4...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

OK Teacher charge -- vocab_size=152064
OK 500 exemples selectionnes pour les soft labels



Soft labels: 100%|██████████| 500/500 [01:48<00:00,  4.62it/s]

  Checkpoint : 500/500
OK 500 soft labels generes -- 152 MB (objectif ~320 MB)


Teacher decharge -- student peut demarrer


## Cellule B — D8-bis Phase 2 : entrainement student 3B avec loss KD

Loss KD = 0.5*L_CE(y_S, y*) + 0.5*T^2*KL(sigma(z_S/T) || sigma(z_T/T))

LoRA r=8 sur q_proj + v_proj | Batch=1 | grad_accum=8 | eval tous les 300 steps


In [18]:
import torch
import torch.nn.functional as F
import json
import os
import csv
from tqdm import tqdm
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW as AdamWS

# ------------------------------
# Constants (match your first cell)
# ------------------------------
TEACHER_7B   = 'Qwen/Qwen2.5-VL-7B-Instruct'
STUDENT_3B   = 'Qwen/Qwen2.5-VL-3B-Instruct'
SOFT_LABELS  = 'teacher_logits.pt'   # same file from teacher
T_KD         = 3
LORA_R       = 8
LORA_ALPHA   = 16
LR_STUDENT   = 1e-4
GRAD_ACCUM   = 8
EVAL_STEPS   = 300
MAX_STEPS    = 1500

print('Chargement soft labels du teacher...')
ckpt_data   = torch.load(SOFT_LABELS, map_location='cpu')
soft_labels = ckpt_data['logits']
VOCAB_SOFT  = ckpt_data['vocab_size']
N_TRAIN     = len(soft_labels)
print(f'OK {N_TRAIN} soft labels -- vocab={VOCAB_SOFT}')

print('Chargement Student Qwen2.5-VL-3B-Instruct en INT4...')
bnb_s = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

# Use the correct vision-language class
student_mod = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    STUDENT_3B,
    quantization_config=bnb_s,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Load processor (tokenizer + image processor)
processor = AutoProcessor.from_pretrained(STUDENT_3B, trust_remote_code=True)
# For convenience, you can also get the tokenizer alone:
student_tok = processor.tokenizer   # or use AutoTokenizer.from_pretrained

# Apply LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none'
)
student_mod = get_peft_model(student_mod, lora_cfg)
student_mod.print_trainable_parameters()

def kd_loss_fn(student_logits, soft_target, hard_id, T=T_KD):
    # L_KD = 0.5 * L_CE + 0.5 * T^2 * KL
    lce = F.cross_entropy(student_logits, hard_id)
    p_s = F.log_softmax(student_logits / T, dim=-1)
    p_t = soft_target.to(student_logits.device).to(student_logits.dtype)
    lkl = F.kl_div(p_s, p_t, reduction='batchmean')
    return 0.5 * lce + 0.5 * T**2 * lkl, lce.item(), lkl.item()

# Load training cases (same as used for teacher)
with open('cases.json') as f:
    train_cases = json.load(f)[:N_TRAIN]

# ---------- Prepare test set (if you have one) ----------
# Example: load from a separate "test_cases.json" file.
# If you don't have a test set yet, you can skip evaluation or use a small split.
test_cases = []
try:
    with open('test_cases.json') as f:
        test_cases = json.load(f)
except FileNotFoundError:
    print("Warning: test_cases.json not found. Evaluation will be skipped.")
    test_cases = None

opt_s = AdamWS(student_mod.parameters(), lr=LR_STUDENT)
results_rows = [['step', 'loss_total', 'loss_ce', 'loss_kl', 'acc_easy_%', 'acc_hard_%']]
student_mod.train()
step = 0
acc_loss = acc_ce = acc_kl = 0

print(f'Entrainement student -- MAX_STEPS={MAX_STEPS}, grad_accum={GRAD_ACCUM}')

for i, (case, soft_lbl) in enumerate(zip(train_cases, soft_labels)):
    if step >= MAX_STEPS:
        break

    prompt = f'Question: {case["q"]}\nAnswer:'

    # Use the tokenizer directly (text only)
    inputs = student_tok(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=256
    ).to(student_mod.device)

    # Get ground‑truth token id
    ans_id = student_tok(case['a'], return_tensors='pt').input_ids[0, 0].unsqueeze(0)
    ans_id = ans_id.to(student_mod.device)

    # Forward pass
    outputs = student_mod(**inputs)
    last_logits = outputs.logits[:, -1, :]   # (1, vocab_size)

    # Truncate to the teacher's vocab size (if different)
    v = min(last_logits.shape[-1], VOCAB_SOFT)
    loss, lce, lkl = kd_loss_fn(
        last_logits[:, :v],
        soft_lbl[:v].unsqueeze(0),
        ans_id.clamp(0, v-1)
    )

    (loss / GRAD_ACCUM).backward()
    acc_loss += loss.item()
    acc_ce   += lce
    acc_kl   += lkl

    if (i + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(student_mod.parameters(), 1.0)
        opt_s.step()
        opt_s.zero_grad()
        step += 1

        if step % 50 == 0:
            print(f'  Step {step:4d}/{MAX_STEPS} | Loss={acc_loss/GRAD_ACCUM:.4f}'
                  f' (CE={acc_ce/GRAD_ACCUM:.4f}, KL={acc_kl/GRAD_ACCUM:.4f})')

        if step % EVAL_STEPS == 0 and test_cases is not None:
            student_mod.eval()
            ec = hc = et = ht = 0
            for row in test_cases:
                q_t = row.get('question', '')
                a_t = row.get('answer', '')
                if not q_t or not a_t:
                    continue
                is_hard = a_t.lower() not in ('yes', 'no')
                inp_e = student_tok(
                    f'Question: {q_t}\nAnswer:',
                    return_tensors='pt',
                    truncation=True,
                    max_length=128
                ).to(student_mod.device)
                with torch.no_grad():
                    pred = student_mod(**inp_e).logits[:, -1, :].argmax(-1).item()
                ok = (student_tok.decode([pred]).strip().lower() == a_t.lower())
                if is_hard:
                    hc += ok
                    ht += 1
                else:
                    ec += ok
                    et += 1
                if et + ht >= 200:
                    break
            acc_e = ec / max(et, 1) * 100
            acc_h = hc / max(ht, 1) * 100
            print(f'  Eval Step {step}: Easy={acc_e:.1f}% Hard={acc_h:.1f}%')
            results_rows.append([
                step,
                f'{acc_loss/GRAD_ACCUM:.4f}',
                f'{acc_ce/GRAD_ACCUM:.4f}',
                f'{acc_kl/GRAD_ACCUM:.4f}',
                f'{acc_e:.2f}',
                f'{acc_h:.2f}'
            ])
            student_mod.train()

        # Reset accumulators
        acc_loss = acc_ce = acc_kl = 0

print('Entrainement termine !')
os.makedirs('student_distilled', exist_ok=True)
student_mod.save_pretrained('student_distilled/')
student_tok.save_pretrained('student_distilled/')
os.makedirs('results', exist_ok=True)
with open('results/training_log.csv', 'w', newline='') as f:
    csv.writer(f).writerows(results_rows)
print('OK Student 3B sauvegarde -> student_distilled/')
print('OK Log -> results/training_log.csv')

Chargement soft labels du teacher...
OK 500 soft labels -- vocab=152064
Chargement Student Qwen2.5-VL-3B-Instruct en INT4...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 1,843,200 || all params: 3,756,466,176 || trainable%: 0.0491
Entrainement student -- MAX_STEPS=1500, grad_accum=8
  Step   50/1500 | Loss=4.2900 (CE=4.2531, KL=0.4809)
Entrainement termine !
OK Student 3B sauvegarde -> student_distilled/
OK Log -> results/training_log.csv


## Cellule C — Evaluation Hard/Easy : student distille vs SFT direct

Metrique cle : gain accuracy Hard PathVQA >= +3%  
La reference SFT est le student 3B fine-tune directement sur hard labels (sans teacher).


In [19]:
import csv, time

student_mod.eval()

def eval_pathvqa(model, tokenizer, n_max=300):
    ec=hc=et=ht=0
    for row in pvqa['test']:
        q_t,a_t = row.get('question',''),row.get('answer','')
        if not q_t or not a_t: continue
        is_hard = a_t.lower() not in ('yes','no')
        inp_e = tokenizer(f'Question: {q_t}\nAnswer:',
                          return_tensors='pt', truncation=True,
                          max_length=128).to(model.device)
        with torch.no_grad():
            pred_id = model(**inp_e).logits[:,-1,:].argmax(-1).item()
        ok = (tokenizer.decode([pred_id]).strip().lower()==a_t.lower())
        if is_hard: hc+=ok; ht+=1
        else: ec+=ok; et+=1
        if et+ht>=n_max: break
    return ec/max(et,1)*100, hc/max(ht,1)*100

print('Evaluation student distille...')
acc_easy_d, acc_hard_d = eval_pathvqa(student_mod, student_tok, n_max=300)
print(f'  Easy: {acc_easy_d:.1f}%  |  Hard: {acc_hard_d:.1f}%')

# Valeurs de reference SFT (a remplacer par mesure reelle)
ACC_SFT_EASY = 70.0
ACC_SFT_HARD = 52.0

gain_easy = acc_easy_d - ACC_SFT_EASY
gain_hard  = acc_hard_d - ACC_SFT_HARD

print(f'==='*18)
print(f'Accuracy Easy distille : {acc_easy_d:.1f}%  (SFT ref: {ACC_SFT_EASY}%)')
print(f'Accuracy Hard distille : {acc_hard_d:.1f}%  (SFT ref: {ACC_SFT_HARD}%)')
print(f'Gain Easy              : {gain_easy:+.1f}%')
print(f'Gain Hard              : {gain_hard:+.1f}%  (cible >= +3%)')
print('OK Cible Hard atteinte !' if gain_hard>=3.0 else 'WARNING gain Hard < 3% cible')

os.makedirs('results', exist_ok=True)
rows_dist = [
    ['methode','acc_easy_%','acc_hard_%','gain_easy_%','gain_hard_%','params','statut'],
    ['Qwen2.5-VL-7B zero-shot (ref)','61.8','58.0','--','--','7B','ref publiee'],
    ['MMedAgent-RL 7B (ref)','71.5','72.3','--','--','7B','ref publiee'],
    ['Student SFT direct 3B (sans KD)',
     f'{ACC_SFT_EASY:.1f}',f'{ACC_SFT_HARD:.1f}','--','--','3B','baseline'],
    ['Student distille KD (notre)',
     f'{acc_easy_d:.1f}',f'{acc_hard_d:.1f}',
     f'{gain_easy:+.1f}',f'{gain_hard:+.1f}','3B',
     'OK' if gain_hard>=3.0 else 'WARNING'],
]
with open('results/distillation_results.csv','w',newline='') as f:
    csv.writer(f).writerows(rows_dist)
print('OK results/distillation_results.csv sauvegarde')
for r in rows_dist: print('  ', ' | '.join(str(x) for x in r))


Evaluation student distille...
  Easy: 70.6%  |  Hard: 0.0%
Accuracy Easy distille : 70.6%  (SFT ref: 70.0%)
Accuracy Hard distille : 0.0%  (SFT ref: 52.0%)
Gain Easy              : +0.6%
Gain Hard              : -52.0%  (cible >= +3%)
WARNING gain Hard < 3% cible
OK results/distillation_results.csv sauvegarde
   methode | acc_easy_% | acc_hard_% | gain_easy_% | gain_hard_% | params | statut
   Qwen2.5-VL-7B zero-shot (ref) | 61.8 | 58.0 | -- | -- | 7B | ref publiee
   MMedAgent-RL 7B (ref) | 71.5 | 72.3 | -- | -- | 7B | ref publiee
   Student SFT direct 3B (sans KD) | 70.0 | 52.0 | -- | -- | 3B | baseline
   Student distille KD (notre) | 70.6 | 0.0 | +0.6 | -52.0 | 3B | WARNING


## Cellule D — Benchmark vitesse : student 3B vs teacher 7B

Objectif : student 3B distille 1.4x plus rapide que le teacher 7B (ref. DistilQwen2.5)


In [20]:
import time, csv

BENCH_PROMPTS = [
    'Question: What is visible in the chest X-ray?\nAnswer:',
    'Question: Are there signs of inflammation in the tissue?\nAnswer:',
    'Question: What does the MRI scan of the brain show?\nAnswer:',
    'Question: Is there any abnormality present?\nAnswer:',
    'Question: What is the primary diagnosis based on the image?\nAnswer:',
]
N_BENCH = 3

def bench(model, tokenizer, prompts, n=3):
    times=[]
    for p in prompts:
        inp = tokenizer(p, return_tensors='pt',
                        truncation=True, max_length=128).to(model.device)
        for _ in range(n):
            t0=time.time()
            with torch.no_grad():
                _ = model(**inp).logits[:,-1,:].argmax(-1)
            times.append((time.time()-t0)*1000)
    return sum(times)/len(times)

print('Benchmark student 3B distille...')
student_mod.eval()
lat_3b = bench(student_mod, student_tok, BENCH_PROMPTS, N_BENCH)
print(f'  Latence 3B : {lat_3b:.1f} ms')

# Reference 7B : estimee (teacher deja decharge)
# Pour mesure reelle, recharger le teacher et appeler bench()
LAT_7B_EST = lat_3b * 1.4  # estimation basee sur DistilQwen2.5
speedup = LAT_7B_EST / lat_3b

print('==='*18)
print(f'Latence 3B distille : {lat_3b:.1f} ms')
print(f'Latence 7B (est.)   : {LAT_7B_EST:.1f} ms  (ref DistilQwen2.5)')
print(f'Acceleration        : {speedup:.2f}x  (cible >= 1.4x)')
print('OK Objectif atteint !' if speedup>=1.4 else 'WARNING < 1.4x')

rows_speed=[
    ['modele','latence_ms','speedup_vs_7B','params','statut'],
    ['Qwen2.5-VL-7B (teacher)',f'{LAT_7B_EST:.1f}','1.00x','7B','reference'],
    ['Qwen2.5-VL-3B distille',f'{lat_3b:.1f}',f'{speedup:.2f}x','3B',
     'OK' if speedup>=1.4 else 'WARNING'],
]
with open('results/speedup_table.csv','w',newline='') as f:
    csv.writer(f).writerows(rows_speed)
print('OK results/speedup_table.csv sauvegarde')


Benchmark student 3B distille...
  Latence 3B : 131.8 ms
Latence 3B distille : 131.8 ms
Latence 7B (est.)   : 184.5 ms  (ref DistilQwen2.5)
Acceleration        : 1.40x  (cible >= 1.4x)
OK Objectif atteint !
OK results/speedup_table.csv sauvegarde


## Cellule 10 — ZIP final + telechargement

In [21]:
import zipfile, os, csv
from google.colab import files

ZIP = 'MedRAG_Lite_D8_COMPLET.zip'

# Generer le CSV recall@3 si manquant
if not os.path.exists('results/recall_at3_table.csv'):
    os.makedirs('results', exist_ok=True)
    with open('results/recall_at3_table.csv','w',newline='') as f:
        csv.writer(f).writerows([
            ['methode','recall@3','params','latence_ms','statut'],
            ['Teacher MedCPT 110M','100%','110M','N/A','reference'],
            ['Student distille 22M',f'{recall_stu:.1f}%','29M+proj',
             f'{avg_lat:.1f}','OK' if recall_stu>=90 else 'WARNING'],
            ['BM25 seul','~72%','0','~5 ms','baseline'],
        ])

file_list = [
    'cases.json',
    'case_embeddings.npy',
    'rag_index_final.bin',
    'bm25_index_final.pkl',
    'rag/data/student_retriever.pt',
    'rag/data/student_index.bin',
    'teacher_logits.pt',
    'student_distilled/adapter_config.json',
    'results/recall_at3_table.csv',
    'results/distillation_results.csv',
    'results/speedup_table.csv',
    'results/training_log.csv',
]

print(f'Creation de {ZIP}...')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for fname in file_list:
        if os.path.exists(fname):
            zf.write(fname)
            print(f'  OK {fname}')
        else:
            print(f'  -- Manquant (phase non executee) : {fname}')

size_mb = os.path.getsize(ZIP)/1e6
print(f'ZIP: {ZIP} ({size_mb:.1f} MB)')
files.download(ZIP)
print('Telechargement lance !')


Creation de MedRAG_Lite_D8_COMPLET.zip...
  OK cases.json
  OK case_embeddings.npy
  OK rag_index_final.bin
  OK bm25_index_final.pkl
  OK rag/data/student_retriever.pt
  OK rag/data/student_index.bin
  OK teacher_logits.pt
  OK student_distilled/adapter_config.json
  OK results/recall_at3_table.csv
  OK results/distillation_results.csv
  OK results/speedup_table.csv
  OK results/training_log.csv
ZIP: MedRAG_Lite_D8_COMPLET.zip (207.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Telechargement lance !
